# Phase Angle Correspondence

**Hypothesis:** Each neuron within a frequency group encodes that frequency at a different
phase offset. The ring structure in W_in PCA space (PC1 vs PC2) is the geometric signature
of this phase tiling.

This notebook tests three measurements of the same putative phase angle:

1. **PCA angle** — `atan2(PC2, PC1)` from projecting W_in columns onto the group's
   reference eigenbasis. This is a purely geometric observation.

2. **Weight Fourier phase** (`φ_mk`) — phase of `θ_m = W_E[:p] @ W_in[:, m]` in the
   Fourier basis at the group's dominant frequency. This measures what frequency/phase
   each neuron's input weights are tuned to respond to *in token space*.

3. **Activation Fourier phase** — phase of the 2D DFT of `neuron_activations[n]` at
   the (f, f) bin, capturing the `cos(2πf*(a+b)/p + φ)` component of actual output.

If all three agree (up to a global rotation), it means:
- The ring in W_in space IS the Fourier phase structure
- The linear weight phase propagates through nonlinearity into activations
- Neurons are genuinely tiling phase space, not just occupying a ring by accident

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from miscope import load_family
from miscope.analysis.artifact_loader import ArtifactLoader

In [2]:
FAMILY_NAME = "modulo_addition_1layer"
EXPORT_DIR = Path("exports/phase_angle_correspondence")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# (prime, model_seed, data_seed, label)
MODELS = [
    (109, 485, 598, "p109 reference healthy"),
    (59,  485, 598, "p59 incomplete freq set"),
    (113, 999, 598, "p113 canon (diffuse)"),
    (101, 999, 598, "p101 open loop"),
]

family = load_family(FAMILY_NAME)

## Data loading helpers

In [3]:
def load_phase_data(variant, epoch=None):
    """Load all artifacts needed for phase comparison at a given epoch.

    If epoch is None, uses the final available epoch.

    Returns dict with:
        epoch         — actual epoch used
        group_bases   — (n_groups, 3, d_model) from neuron_group_pca cross_epoch
        group_freqs   — (n_groups,) frequency index per group
        group_sizes   — (n_groups,) neuron count at reference epoch
        W_in          — (d_model, d_mlp) current-epoch weights
        norm_matrix   — (n_freq, d_mlp) current-epoch frequency norms
        phi_mk        — (d_mlp, K) Fourier phases of effective input weights
        alpha_mk      — (d_mlp, K) Fourier magnitudes (for purity filtering)
        freq_indices  — (K,) frequency values 1..K
        activations   — (d_mlp, p, p) MLP activation grid
        prime         — int
    """
    loader = ArtifactLoader(str(variant.variant_dir / "artifacts"))
    epochs = sorted(loader.get_epochs("parameter_snapshot"))
    if epoch is None:
        epoch = epochs[-1]

    cross = loader.load_cross_epoch("neuron_group_pca")
    snap = loader.load_epoch("parameter_snapshot", epoch)
    norm_data = loader.load_epoch("neuron_freq_norm", epoch)
    fourier = loader.load_epoch("neuron_fourier", epoch)
    acts_data = loader.load_epoch("neuron_activations", epoch)

    return {
        "epoch": epoch,
        "group_bases": cross["group_bases"],
        "group_freqs": cross["group_freqs"],
        "group_sizes": cross["group_sizes"],
        "W_in": snap["W_in"],
        "norm_matrix": norm_data["norm_matrix"],
        "phi_mk": fourier["phi_mk"],
        "alpha_mk": fourier["alpha_mk"],
        "freq_indices": fourier["freq_indices"],
        "activations": acts_data["activations"],
        "prime": int(variant.model_config["prime"]),
    }

## Per-group phase extraction

In [4]:
def extract_group_phases(data, g_idx):
    """Extract all three phase measurements for one frequency group.

    Args:
        data:  output of load_phase_data
        g_idx: group index into group_freqs / group_bases

    Returns dict with:
        members      — (n_group,) neuron indices in this group
        freq         — int, dominant frequency
        pca_angle    — (n_group,) atan2(PC2, PC1) in (-π, π]
        weight_phase — (n_group,) phi_mk at group frequency
        act_phase    — (n_group,) 2D DFT phase at (f, f)
        weight_mag   — (n_group,) alpha_mk at group frequency (for filtering)
    """
    freq = int(data["group_freqs"][g_idx])
    dominant_freq = np.argmax(data["norm_matrix"], axis=0)  # (d_mlp,)
    members = np.where(dominant_freq == freq)[0]

    # 1. PCA angle: project W_in columns onto reference basis
    basis = data["group_bases"][g_idx]  # (3, d_model)
    coords = basis @ data["W_in"][:, members]  # (3, n_group)
    pca_angle = np.arctan2(coords[1], coords[0])  # (n_group,)

    # 2. Weight Fourier phase: phi_mk at group frequency
    # freq_indices are [1, 2, ..., K] → freq f maps to column f-1
    k_idx = freq - 1
    weight_phase = data["phi_mk"][members, k_idx]  # (n_group,)
    weight_mag = data["alpha_mk"][members, k_idx]  # (n_group,)

    # 3. Activation Fourier phase: 2D DFT at (f, f)
    acts_group = data["activations"][members]  # (n_group, p, p)
    fft2d = np.fft.fft2(acts_group, axes=(1, 2))  # (n_group, p, p)
    act_phase = np.angle(fft2d[:, freq, freq])  # (n_group,)

    return {
        "members": members,
        "freq": freq,
        "pca_angle": pca_angle,
        "weight_phase": weight_phase,
        "act_phase": act_phase,
        "weight_mag": weight_mag,
    }

## Circular statistics helpers

Phases are angles — ordinary correlation doesn't work because -π and π are the same
point. We use:

- **Angular residual**: subtract the best-fit global rotation and look at the spread
- **Mean resultant length R**: how tightly the angular differences cluster around zero
  (R=1 = perfect agreement, R=0 = uniform spread)
- **Circular-linear correlation**: Jammalamadaka-SenGupta rank correlation for two
  angle sequences

In [5]:
def angular_residual_stats(angle_a, angle_b):
    """Compute agreement between two angle arrays (circular, same period).

    Finds the global rotation δ that best aligns angle_b to angle_a,
    then returns the residual spread.

    Returns:
        delta     — best-fit global rotation (radians)
        residuals — (n,) wrapped angular differences after alignment
        R         — mean resultant length of residuals (0=uniform, 1=perfect)
    """
    # Find the mean of (angle_a - angle_b) in circular sense
    diff = angle_a - angle_b
    delta = np.arctan2(np.sin(diff).mean(), np.cos(diff).mean())
    residuals = np.arctan2(np.sin(diff - delta), np.cos(diff - delta))
    R = float(np.sqrt(np.sin(residuals).mean()**2 + np.cos(residuals).mean()**2))
    return delta, residuals, R


def wrap(angles):
    """Wrap angles to (-π, π]."""
    return np.arctan2(np.sin(angles), np.cos(angles))

## Core comparison: PCA angle vs weight Fourier phase

For each frequency group: scatter of PCA angle (x) vs weight Fourier phase φ_mk (y).

A diagonal relationship (with possible rotation) → the two measurements agree.
Random scatter → the ring in W_in space is not aligned with the Fourier basis.

In [6]:
def render_phase_scatter(phases, label, min_weight_mag=0.0):
    """Scatter plot: PCA angle vs weight Fourier phase, all groups in one figure.

    Args:
        phases:         list of group-phase dicts from extract_group_phases
        label:          model label for title
        min_weight_mag: threshold on alpha_mk for including a neuron (filters
                        neurons with weak frequency sensitivity)
    """
    import colorsys

    n_groups = len(phases)
    fig = go.Figure()

    summary_rows = []

    for g_idx, gp in enumerate(phases):
        mask = gp["weight_mag"] >= min_weight_mag
        if mask.sum() < 2:
            continue

        pca_a = gp["pca_angle"][mask]
        wt_a = gp["weight_phase"][mask]

        delta, residuals, R = angular_residual_stats(pca_a, wt_a)
        summary_rows.append((gp["freq"], len(gp["members"]), mask.sum(), R, np.degrees(delta)))

        hue = g_idx / max(n_groups, 1)
        r, g, b = colorsys.hls_to_rgb(hue, 0.5, 0.65)
        color = f"rgba({int(r*255)},{int(g*255)},{int(b*255)},0.7)"

        fig.add_trace(go.Scatter(
            x=pca_a.tolist(),
            y=wt_a.tolist(),
            mode="markers",
            name=f"freq {gp['freq']} (n={mask.sum()}, R={R:.2f})",
            marker=dict(color=color, size=5),
            hovertemplate="PCA angle=%{x:.3f}<br>φ_mk=%{y:.3f}<extra></extra>",
        ))

    # Diagonal reference
    fig.add_trace(go.Scatter(
        x=[-np.pi, np.pi],
        y=[-np.pi, np.pi],
        mode="lines",
        line=dict(color="rgba(0,0,0,0.2)", width=1, dash="dash"),
        showlegend=False,
        hoverinfo="skip",
    ))

    fig.update_layout(
        title=f"{label} — PCA angle vs weight Fourier phase (φ_mk)<br>"
              "<sup>Diagonal = perfect agreement | R = mean resultant length after alignment</sup>",
        xaxis_title="PCA angle atan2(PC2, PC1)  [rad]",
        yaxis_title="Weight Fourier phase φ_mk  [rad]",
        xaxis=dict(range=[-np.pi - 0.2, np.pi + 0.2]),
        yaxis=dict(range=[-np.pi - 0.2, np.pi + 0.2], scaleanchor="x"),
        template="plotly_white",
        height=540,
        margin=dict(l=70, r=20, t=80, b=60),
        legend=dict(orientation="h", y=-0.15, font=dict(size=10)),
    )

    # Print summary
    print(f"\n{label}")
    print(f"  {'freq':>5}  {'n_grp':>6}  {'n_used':>6}  {'R':>6}  {'δ(deg)':>8}")
    for row in summary_rows:
        print(f"  {row[0]:>5}  {row[1]:>6}  {row[2]:>6}  {row[3]:>6.3f}  {row[4]:>8.1f}")

    return fig


for prime, seed, data_seed, label in MODELS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    data = load_phase_data(v)
    phases = [extract_group_phases(data, g) for g in range(len(data["group_freqs"]))]
    fig = render_phase_scatter(phases, label)
    fig.show()
    tag = f"p{prime}_s{seed}_ds{data_seed}"
    fig.write_image(str(EXPORT_DIR / f"{tag}_pca_vs_weight_phase.png"))


p109 reference healthy
   freq   n_grp  n_used       R    δ(deg)
      3     188     188   0.661      -0.4
     13     127     127   0.195    -113.8
     26     197     197   0.232    -151.8



p59 incomplete freq set
   freq   n_grp  n_used       R    δ(deg)
      4     153     153   0.519      92.3
     20     359     359   0.382     -78.3



p113 canon (diffuse)
   freq   n_grp  n_used       R    δ(deg)
      8     107     107   0.609     -59.9
     32     161     161   0.563    -153.7
     37      78      78   0.620      52.4
     54     166     166   0.235    -134.0



p101 open loop
   freq   n_grp  n_used       R    δ(deg)
     34     109     109   0.417      -9.9
     40     167     167   0.353     -26.4
     42     141     141   0.202    -101.4
     43      95      95   0.453     158.6


## Polar histograms: are phases uniformly distributed?

If the ring hypothesis is correct, PCA angles should be **uniformly distributed** around
the circle. Spokes in the scatter correspond to clustering — distinct preferred phases
rather than a continuous ring.

In [8]:
def render_polar_histogram(phases, label, n_bins=36):
    """Polar histogram of PCA angles for each frequency group."""
    import colorsys

    n_groups = len(phases)
    cols = min(n_groups, 3)
    rows = (n_groups + cols - 1) // cols
    subplot_titles = [f"freq {gp['freq']} (n={len(gp['members'])})" for gp in phases]

    fig = make_subplots(
        rows=rows, cols=cols,
        specs=[[{"type": "polar"}] * cols for _ in range(rows)],
        subplot_titles=subplot_titles,
    )

    bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

    for idx, gp in enumerate(phases):
        row, col = divmod(idx, cols)
        hue = idx / max(n_groups, 1)
        r, g, b = colorsys.hls_to_rgb(hue, 0.5, 0.65)
        color = f"rgba({int(r*255)},{int(g*255)},{int(b*255)},0.7)"

        counts, _ = np.histogram(gp["pca_angle"], bins=bin_edges)

        fig.add_trace(
            go.Barpolar(
                r=counts.tolist(),
                theta=np.degrees(bin_centers).tolist(),
                width=(360 / n_bins),
                marker_color=color,
                showlegend=False,
            ),
            row=row + 1, col=col + 1,
        )

    fig.update_layout(
        title=f"{label} — PCA angle polar histograms per group<br>"
              "<sup>Uniform distribution = phase tiling | Spokes = discrete preferred phases</sup>",
        template="plotly_white",
        height=380 * rows,
        margin=dict(l=40, r=40, t=80, b=40),
    )
    return fig


for prime, seed, data_seed, label in MODELS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    data = load_phase_data(v)
    phases = [extract_group_phases(data, g) for g in range(len(data["group_freqs"]))]
    fig = render_polar_histogram(phases, label)
    fig.show()
    tag = f"p{prime}_s{seed}_ds{data_seed}"
    fig.write_image(str(EXPORT_DIR / f"{tag}_polar_hist.png"))

## Activation Fourier phase vs weight Fourier phase

Now we bring in the actual activation output. `neuron_activations` is `(d_mlp, p, p)` —
the activation for each neuron across all (a, b) input pairs.

For a neuron encoding `cos(2πf*(a+b)/p + φ)`, the 2D DFT at bin (f, f) gives a complex
number whose angle is φ (the activation-space phase).

We test: does the activation phase agree with φ_mk (the weight-space phase)?

In [9]:
def render_act_vs_weight_phase(phases, label):
    """Scatter: activation Fourier phase vs weight Fourier phase φ_mk."""
    import colorsys

    n_groups = len(phases)
    fig = go.Figure()

    print(f"\n{label} — activation phase vs weight phase")
    print(f"  {'freq':>5}  {'n':>6}  {'R_act_wt':>10}  {'R_act_pca':>11}")

    for g_idx, gp in enumerate(phases):
        hue = g_idx / max(n_groups, 1)
        r, g, b = colorsys.hls_to_rgb(hue, 0.5, 0.65)
        color = f"rgba({int(r*255)},{int(g*255)},{int(b*255)},0.7)"

        wt = gp["weight_phase"]
        act = gp["act_phase"]
        pca = gp["pca_angle"]

        _, _, R_aw = angular_residual_stats(act, wt)
        _, _, R_ap = angular_residual_stats(act, pca)
        print(f"  {gp['freq']:>5}  {len(wt):>6}  {R_aw:>10.3f}  {R_ap:>11.3f}")

        fig.add_trace(go.Scatter(
            x=wt.tolist(),
            y=act.tolist(),
            mode="markers",
            name=f"freq {gp['freq']} (R={R_aw:.2f})",
            marker=dict(color=color, size=5),
            hovertemplate="φ_mk=%{x:.3f}<br>act phase=%{y:.3f}<extra></extra>",
        ))

    fig.add_trace(go.Scatter(
        x=[-np.pi, np.pi], y=[-np.pi, np.pi],
        mode="lines",
        line=dict(color="rgba(0,0,0,0.2)", width=1, dash="dash"),
        showlegend=False, hoverinfo="skip",
    ))

    fig.update_layout(
        title=f"{label} — activation phase vs weight Fourier phase φ_mk<br>"
              "<sup>2D DFT at (f, f) bin captures cos(2πf*(a+b)/p + φ) component</sup>",
        xaxis_title="Weight Fourier phase φ_mk  [rad]",
        yaxis_title="Activation 2D-DFT phase at (f,f)  [rad]",
        xaxis=dict(range=[-np.pi - 0.2, np.pi + 0.2]),
        yaxis=dict(range=[-np.pi - 0.2, np.pi + 0.2], scaleanchor="x"),
        template="plotly_white",
        height=540,
        margin=dict(l=70, r=20, t=80, b=60),
        legend=dict(orientation="h", y=-0.15, font=dict(size=10)),
    )
    return fig


for prime, seed, data_seed, label in MODELS:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    data = load_phase_data(v)
    phases = [extract_group_phases(data, g) for g in range(len(data["group_freqs"]))]
    fig = render_act_vs_weight_phase(phases, label)
    fig.show()
    tag = f"p{prime}_s{seed}_ds{data_seed}"
    fig.write_image(str(EXPORT_DIR / f"{tag}_act_vs_weight_phase.png"))


p109 reference healthy — activation phase vs weight phase
   freq       n    R_act_wt    R_act_pca
      3     188       0.036        0.080
     13     127       0.184        0.405
     26     197       0.041        0.044



p59 incomplete freq set — activation phase vs weight phase
   freq       n    R_act_wt    R_act_pca
      4     153       0.107        0.086
     20     359       0.165        0.041



p113 canon (diffuse) — activation phase vs weight phase
   freq       n    R_act_wt    R_act_pca
      8     107       0.171        0.107
     32     161       0.068        0.076
     37      78       0.130        0.029
     54     166       0.068        0.206



p101 open loop — activation phase vs weight phase
   freq       n    R_act_wt    R_act_pca
     34     109       0.177        0.252
     40     167       0.059        0.128
     42     141       0.170        0.062
     43      95       0.140        0.326


## Summary: all three phases per group (best model)

For p109's largest group, show all three angle measurements side by side:
sorted by PCA angle, with weight phase and activation phase overlaid.

If all three form the same monotone curve, the three measurements are all
reading the same underlying phase offset — the neuron's position on the ring.

In [10]:
def render_phase_tripanel(gp, label):
    """Three-panel: sorted PCA angle, weight phase, activation phase.

    Neurons are sorted by PCA angle. If weight phase and activation phase
    follow the same rank order, they're measuring the same thing.
    """
    sort_idx = np.argsort(gp["pca_angle"])
    pca_sorted = gp["pca_angle"][sort_idx]

    # Align weight phase and act phase to match pca sort order
    wt_aligned = wrap(gp["weight_phase"][sort_idx] -
                      np.arctan2(np.sin(gp["weight_phase"] - gp["pca_angle"]).mean(),
                                 np.cos(gp["weight_phase"] - gp["pca_angle"]).mean()))
    act_aligned = wrap(gp["act_phase"][sort_idx] -
                       np.arctan2(np.sin(gp["act_phase"] - gp["pca_angle"]).mean(),
                                  np.cos(gp["act_phase"] - gp["pca_angle"]).mean()))

    neuron_idx = np.arange(len(sort_idx))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=neuron_idx.tolist(), y=pca_sorted.tolist(),
        mode="markers", name="PCA angle",
        marker=dict(color="steelblue", size=5),
    ))
    fig.add_trace(go.Scatter(
        x=neuron_idx.tolist(), y=wt_aligned.tolist(),
        mode="markers", name="φ_mk (aligned)",
        marker=dict(color="darkorange", size=5, symbol="cross"),
    ))
    fig.add_trace(go.Scatter(
        x=neuron_idx.tolist(), y=act_aligned.tolist(),
        mode="markers", name="act phase (aligned)",
        marker=dict(color="green", size=5, symbol="diamond"),
    ))

    fig.update_layout(
        title=f"{label} — freq {gp['freq']}: all three phases sorted by PCA angle<br>"
              "<sup>Overlapping curves → all three measurements agree on neuron ordering</sup>",
        xaxis_title="Neuron rank (sorted by PCA angle)",
        yaxis_title="Phase [rad]",
        yaxis=dict(range=[-np.pi - 0.3, np.pi + 0.3]),
        template="plotly_white",
        height=450,
        margin=dict(l=70, r=20, t=80, b=60),
        legend=dict(orientation="h", y=-0.15, font=dict(size=10)),
    )
    return fig


# Show for p109 (clearest) and p101 (messiest) for contrast
for prime, seed, data_seed, label in [
    (109, 485, 598, "p109 reference healthy"),
    (101, 999, 598, "p101 open loop"),
]:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    data = load_phase_data(v)
    phases = [extract_group_phases(data, g) for g in range(len(data["group_freqs"]))]
    # Largest group
    largest = max(phases, key=lambda gp: len(gp["members"]))
    fig = render_phase_tripanel(largest, label)
    fig.show()
    tag = f"p{prime}_s{seed}_ds{data_seed}"
    fig.write_image(str(EXPORT_DIR / f"{tag}_tripanel.png"))

## Sanity check: activation heatmaps sorted by PCA phase angle

For the largest group, show activation heatmaps of the first 12 neurons sorted by
PCA angle. If the phase hypothesis holds, the activation 'hot stripe' should rotate
gradually from one neuron to the next — tracing out the full circle across the group.

In [11]:
def render_activation_grid_sorted(data, gp, n_show=12, label=""):
    """Grid of activation heatmaps sorted by PCA angle.

    Shows the first n_show neurons. If phase is real, the diagonal
    stripe shifts progressively across the grid.
    """
    sort_idx = np.argsort(gp["pca_angle"])
    n_show = min(n_show, len(sort_idx))
    selected = sort_idx[:n_show]
    members_sorted = gp["members"][selected]
    angles_sorted = gp["pca_angle"][selected]

    acts = data["activations"][members_sorted]  # (n_show, p, p)

    cols = 4
    rows = (n_show + cols - 1) // cols

    subplot_titles = [
        f"n={members_sorted[i]} φ={np.degrees(angles_sorted[i]):.0f}°"
        for i in range(n_show)
    ]

    fig = make_subplots(rows=rows, cols=cols, subplot_titles=subplot_titles,
                        horizontal_spacing=0.04, vertical_spacing=0.12)

    for idx in range(n_show):
        row, col = divmod(idx, cols)
        # Normalize each neuron independently
        act = acts[idx]
        vmax = np.abs(act).max()
        if vmax > 1e-8:
            act = act / vmax

        fig.add_trace(
            go.Heatmap(
                z=act.tolist(),
                colorscale="RdBu_r",
                zmin=-1, zmax=1,
                showscale=(idx == 0),
                hoverinfo="skip",
            ),
            row=row + 1, col=col + 1,
        )

    fig.update_layout(
        title=f"{label} — freq {gp['freq']}: activation heatmaps sorted by PCA angle<br>"
              "<sup>Each panel: neuron index and PCA angle | 'hot stripe' should rotate if phase is real</sup>",
        template="plotly_white",
        height=250 * rows,
        margin=dict(l=40, r=40, t=80, b=40),
    )
    # Hide axes on all subplots
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return fig


for prime, seed, data_seed, label in [
    (109, 485, 598, "p109 reference healthy"),
    (59,  485, 598, "p59 incomplete freq set"),
    (101, 999, 598, "p101 open loop"),
]:
    v = family.get_variant(prime=prime, seed=seed, data_seed=data_seed)
    data = load_phase_data(v)
    phases = [extract_group_phases(data, g) for g in range(len(data["group_freqs"]))]
    largest = max(phases, key=lambda gp: len(gp["members"]))
    fig = render_activation_grid_sorted(data, largest, n_show=12, label=label)
    fig.show()
    tag = f"p{prime}_s{seed}_ds{data_seed}"
    fig.write_image(str(EXPORT_DIR / f"{tag}_act_grid_sorted.png"))